In [ ]:
# ==========================================
# BLOCK 1: DATA LOADING AND CLEANING (NLP)
# ==========================================

# 1. Import libraries and load the spaCy language model
import pandas as pd
import spacy

nlp = spacy.load('en_core_web_sm')

# 2. Load the Uber dataset
df_uber = pd.read_csv('../data/uber_reviews_without_reviewid.csv')

# Keep only the columns we are interested in: text and score
df = df_uber[['content', 'score']].copy()

# Drop rows that might be empty by mistake
df = df.dropna(subset=['content'])

# 3. Transform the Score into Sentiment (Our "Class" to predict)
# We will consider Positive (1) if it has 4 or 5 stars, and Negative (0) if it has 3, 2 or 1.
def categorize_sentiment(score):
    if score >= 4:
        return 1 # Positive
    else:
        return 0 # Negative

df['sentiment'] = df['score'].apply(categorize_sentiment)

# 4. Text cleaning function
def clean_text(text):
    # Convert to string in case any numbers slipped in
    doc = nlp(str(text))

    clean_tokens = []
    for word in doc:
        # Filter: no spaces, no punctuation, no stopwords
        if not word.is_space and not word.is_punct and not word.is_stop:
            # Save the word in lowercase and lemmatized
            clean_tokens.append(word.lemma_.lower())

    # Join the clean words back into a single sentence
    return " ".join(clean_tokens)

# 5. Apply the cleaning to all reviews
df['cleaned_content'] = df['content'].apply(clean_text)
print("Cleaning completed!")

# Show the first 5 rows to see the "Before" and "After"
display(df.head())

In [ ]:
# ==========================================
# BLOCK 2: DOCUMENT REPRESENTATION
# ==========================================

import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. BUSINESS VISUALIZATION: WordClouds
# Group all clean text according to sentiment
positive_text = " ".join(df[df['sentiment'] == 1]['cleaned_content'].dropna())
negative_text = " ".join(df[df['sentiment'] == 0]['cleaned_content'].dropna())

# Function to avoid repeating code when drawing clouds
def plot_wordcloud(text, title, colormap):
    # Generate the cloud: 800x400, with a maximum of 100 keywords
    wc = WordCloud(width=800, height=400, background_color='white', colormap=colormap, max_words=100)
    wc.generate(text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.title(title, fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.show()

# Cloud for 4-5 stars
plot_wordcloud(positive_text, "Positive Reviews (4-5 Stars)", "Greens")

# Cloud for 1-3 stars
plot_wordcloud(negative_text, "Negative Reviews (1-3 Stars)", "Reds")


# 2. MATHEMATICAL REPRESENTATION: TF-IDF Matrix

# Initialize the vectorizer keeping the 1000 most relevant words
tfidf = TfidfVectorizer(max_features=1000)

# Train the vectorizer and transform our clean text
# X will be our matrix of independent variables (Features)
X = tfidf.fit_transform(df['cleaned_content'].dropna())

# y will be our dependent variable (Target)
y = df.dropna(subset=['cleaned_content'])['sentiment']

print(f"Shape of the X matrix (Rows, Columns): {X.shape}")

In [ ]:
# ==============================================
# SUB-BLOCK 3.1: LOGISTIC REGRESSION
# ==============================================
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("--- EVALUATING LOGISTIC REGRESSION ---")

# 1. Define the model.
lr_model = LogisticRegression(solver='liblinear', max_iter=1000)

# 2. Stratified Cross-Validation (5 partitions)
# Maintains the real proportion of positive and negative reviews in each test.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# We use cross_validate to get multiple metrics at once
metrics = ['accuracy', 'f1_weighted']
lr_cv_results = cross_validate(lr_model, X, y, cv=skf, scoring=metrics)

print("Cross-Validation Results:")
print(f" -> Accuracy: {lr_cv_results['test_accuracy'].mean() * 100:.2f}%")
print(f" -> F1-Score (Weighted): {lr_cv_results['test_f1_weighted'].mean() * 100:.2f}%\n")

# 3. Training for Visualization
# We split once to get the detailed report and the confusion matrix
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)

print("Classification Report (On 20% Test):")
print(classification_report(y_test, lr_predictions, target_names=['Negative (0)', 'Positive (1)']))

# 4. Visualization: Confusion Matrix
lr_cm = confusion_matrix(y_test, lr_predictions)

plt.figure(figsize=(6, 4))
sns.heatmap(lr_cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title('Logistic Regression Confusion Matrix', fontweight='bold')
plt.show()

In [ ]:
# ==============================================
# SUB-BLOCK 3.2: RANDOM FOREST
# ==============================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import export_graphviz
import graphviz
from IPython.display import display

print("--- EVALUATING RANDOM FOREST ---")

# 1. Define the model
# n_estimators=100 means we will create a "forest" of 100 decision trees
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# 2. Stratified Cross-Validation
# We reuse the 'skf' and 'metrics' variables from Sub-block 3.1
rf_cv_results = cross_validate(rf_model, X, y, cv=skf, scoring=metrics)

print("Stratified Cross-Validation Results:")
print(f" -> Accuracy: {rf_cv_results['test_accuracy'].mean() * 100:.2f}%")
print(f" -> F1-Score (Weighted): {rf_cv_results['test_f1_weighted'].mean() * 100:.2f}%\n")

# 3. Traditional training for Report and Visualization
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

print("Classification Report (On 20% Test):")
print(classification_report(y_test, rf_predictions, target_names=['Negative (0)', 'Positive (1)']))

# 4. Visualization: Decision Trees
# We extract the vocabulary of the 1000 words to name the rules of the trees
word_names = tfidf.get_feature_names_out()

for i in range(3):
    print(f"\n--- Tree {i+1} ---")
    tree = rf_model.estimators_[i]
    dot_data = export_graphviz(tree,
                               feature_names=word_names,
                               class_names=['Negative', 'Positive'],
                               filled=True,
                               max_depth=2, # We show only the first 2 levels to make it readable
                               impurity=False,
                               proportion=True)
    graph = graphviz.Source(dot_data)
    display(graph)

In [ ]:
# ==============================================
# SUB-BLOCK 3.3: SUPPORT VECTOR MACHINE (SVM)
# ==============================================
from sklearn import svm

print("--- EVALUATING SUPPORT VECTOR MACHINE (SVM) ---")

# 1. Define the model
svm_model = svm.SVC(kernel='linear', random_state=42)

# 2. Stratified Cross-Validation
# We continue using 'skf' and 'metrics' from previous blocks
svm_cv_results = cross_validate(svm_model, X, y, cv=skf, scoring=metrics)

print("Stratified Cross-Validation Results:")
print(f" -> Accuracy: {svm_cv_results['test_accuracy'].mean() * 100:.2f}%")
print(f" -> F1-Score (Weighted): {svm_cv_results['test_f1_weighted'].mean() * 100:.2f}%\n")

# 3. Traditional training for Report and Matrix
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)

print("Classification Report (On 20% Test):")
print(classification_report(y_test, svm_predictions, target_names=['Negative (0)', 'Positive (1)']))

# 4. Visualization: Confusion Matrix
svm_cm = confusion_matrix(y_test, svm_predictions)

plt.figure(figsize=(6, 4))
sns.heatmap(svm_cm, annot=True, fmt='d', cmap='Oranges', # We use 'Oranges' to differentiate it graphically
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.title('SVM Confusion Matrix', fontweight='bold')
plt.show()